During the previous validation phase, I identified various anomalies, inconsistencies, and business-specific outliers. In this section, I will evaluate each issue individually to determine whether it needs to be corrected, removed, maintained, or documented, considering its potential impact on business understanding and subsequent analysis.

I'm not touching the original (raw) DataFrames in the `data_understanding` section. Instead, I'll reread the data from the CSVs in the cleanup notebook and use the cleaned versions with new variable names. This is because the raw data should always remain unchanged and traceable. This way, I can see what I changed and why.
I'm uploading the raw data as well.

In [1]:
import pandas as pd

customers = pd.read_csv("../data/raw_data/olist/olist_customers_dataset.csv")
geo_locations = pd.read_csv("../data/raw_data/olist/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw_data/olist/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw_data/olist/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw_data/olist/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw_data/olist/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw_data/olist/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw_data/olist/olist_sellers_dataset.csv")
product_category_name_translations = pd.read_csv("../data/raw_data/olist/product_category_name_translation.csv")
closed_deals = pd.read_csv("../data/raw_data/marketing/olist_closed_deals_dataset.csv")
marketing_leads = pd.read_csv("../data/raw_data/marketing/olist_marketing_qualified_leads_dataset.csv")

In [2]:
orders_clean = orders.copy()
products_clean = products.copy()
order_items_clean = order_items.copy()
order_payments_clean = order_payments.copy()
order_reviews_clean = order_reviews.copy()
customers_clean = customers.copy()
sellers_clean = sellers.copy()
geolocation_clean = geo_locations.copy()
category_translation_clean = product_category_name_translations.copy()
marketing_leads_clean = marketing_leads.copy()
closed_deals_clean = closed_deals.copy()

Decision: I did not delete any duplicate records from the products table. Although 695 products share the same descriptive attributes, each record has a unique product_id and represents different product entities. Deleting these records could inadvertently lead to the removal of valid products. Cleaning will not be applied. Because all I have is the information that "they look similar." I cannot say "These are definitely the same product."

In [3]:
products.duplicated(
    subset=[
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
).sum()

np.int64(695)

There are products that weigh 0 but have dimensions. I'm investigating this.

In [4]:
products["product_weight_g"].isna().sum()

np.int64(2)

I'm setting these cells to NaN for these two products that have a weight of 0. Since there are only a few rows, I haven't used any data completion to avoid adding artificial information. I've preserved missing values ​​for transparency.

In [5]:
products_clean.loc[
    products_clean["product_weight_g"] == 0,
    "product_weight_g"
] = pd.NA

In [6]:
products_clean["product_weight_g"].isna().sum()

np.int64(6)

As we saw in the review section, there were 2 records where the number of installments was 0.

In [7]:
(order_payments_clean["payment_installments"] == 0).sum()

np.int64(2)

I'm not leaving these entries as 0 because there's no such concept as "0 installments". Setting them to 1 is also illogical because it would just be an estimate. That's why I'm marking them as NaN.

In [8]:
order_payments_clean.loc[
    order_payments_clean["payment_installments"] == 0,
    "payment_installments"
] = pd.NA

In [9]:
order_payments_clean["payment_installments"].isna().sum()

np.int64(2)

There were 8 orders that were marked as delivered but had an empty delivery date column. However, I'm not performing a cleanup operation here. Because I can't delete the rows, other columns might be needed, changing the status would lead to inaccurate information, and guessing isn't the right approach here. That's why I'm not touching it.

In [10]:
orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

There are also columns with delivery dates that have been cancelled. However, following the same logic, I'm leaving these records as they are. If the number of these records were much larger, different methods would be used depending on the situation.

In [11]:
orders_clean.loc[
    (orders_clean["order_status"] == "canceled") &
    (orders_clean["order_delivered_customer_date"].notna())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00


There were orders where the shipping date appeared to be before the order confirmation date. This time there are 1359 records, which is 1.4% of the total dataset, not a negligible percentage. Therefore, I shouldn't delete them. I can't change the dates because I don't know which one is incorrect. So I'm not making any changes again because there's an anomaly in the business process. I'll review it in EDA.

In [12]:
carrier_before_approval = orders_clean.loc[
    orders_clean["order_delivered_carrier_date"]
    < orders_clean["order_approved_at"]
]

carrier_before_approval

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04 00:00:00
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15 00:00:00
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31 00:00:00
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31 00:00:00
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06 00:00:00
...,...,...,...,...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,fcdd7556401aaa1c980f8b67a69f95dc,delivered,2018-07-02 16:30:02,2018-07-05 16:17:59,2018-07-05 14:11:00,2018-07-10 23:21:47,2018-07-24 00:00:00
99230,78008d03bd8ef7fcf1568728b316553c,043e3254e68daf7256bda1c9c03c2286,delivered,2018-07-03 13:11:13,2018-07-05 16:32:52,2018-07-03 12:57:00,2018-07-10 17:47:39,2018-07-23 00:00:00
99266,76a948cd55bf22799753720d4545dd2d,3f20a07b28aa252d0502fe7f7eb030a9,delivered,2018-01-30 02:41:30,2018-02-04 23:31:46,2018-01-31 18:11:58,2018-03-18 20:08:50,2018-03-02 00:00:00
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,6d63fa86bd2f62908ad328325799152f,delivered,2018-04-20 17:28:40,2018-04-24 19:26:10,2018-04-23 17:18:40,2018-04-28 17:38:42,2018-05-15 00:00:00


NELER BOŞ

I am examining the answer to the question, "In clean dataframes, which columns still have missing values, and how many are there?"

In [13]:
for name, df in {
    "orders": orders_clean,
    "products": products_clean,
    "order_payments": order_payments_clean,
    "customers": customers_clean,
    "order_items": order_items_clean,
    "order_reviews": order_reviews_clean,
    "sellers": sellers_clean,
    "geolocation": geolocation_clean,
    "category_translation": category_translation_clean,
}.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print(f"\n{name}")
        print(missing.sort_values(ascending=False))


orders
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                6
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

order_payments
payment_installments    2
dtype: int64

order_reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64


With the code above, I identified the columns that were still empty. Now I'm evaluating them one by one. There were records where the order confirmation date column was empty. Most of these records were canceled, which is normal, so there's no confirmation date, and therefore I won't touch these records. Some were marked "created," which is also normal, meaning the order was created but the process hasn't started yet. Some were marked "delivered," but the confirmation date was empty. This is only 14 records, or 0.014% of the total orders. The reasons could be loss during ETL, event logging problems, incomplete export, or timestamps not being written to the system. I'm not cleaning up because I can't reliably guess the cause.

In [14]:
orders_clean.loc[
    orders_clean["order_approved_at"].isna(),
    "order_status"
].value_counts(dropna=False)

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

The presence of carrier and customer delivery timestamps suggests that the order lifecycle was completed successfully. Therefore, the missing approval timestamp is more likely due to incomplete timestamp recording or data extraction rather than an actual business process issue.

In [15]:
delivered_missing_approval = orders_clean.loc[
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_approved_at"].isna())
]

delivered_missing_approval[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ]
]

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
5323,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33
16567,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06
19031,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38
22663,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47
23156,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19
26800,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01
38290,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58
39334,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23
48401,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19
61743,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03


I'm now reviewing the records where the shipping date is blank. Records where the order status is unavailable, canceled, invoiced, processing, created, or approved are normal, as those stages precede the shipping stage. However, there are two orders that are marked 'delivered' but have a blank shipping date. There isn't enough information to fill these in correctly, and they only represent a very small portion of the total order. Therefore, I haven't performed any cleanup operation.

In [16]:
orders_clean.loc[
    orders_clean["order_delivered_carrier_date"].isna(),
    "order_status"
].value_counts(dropna=False)

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

Next, we have the rows where the delivery date is blank. Again, for orders with the status "shipped, canceled, unavailable, invoiced, processing, created, approved," it's normal for the delivery date to be blank. However, orders with the status "delivered" should be examined. But even for those, since there isn't enough evidence to fill in the date, I'm leaving it as is.

In [17]:
orders_clean.loc[
    orders_clean["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts(dropna=False)

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

I've examined all the missing values ​​in the orders table, now I'm moving on to the products table. First, there's a group containing 610 missing values ​​(product_category_name, product_name_length, product_description_length, product_photos_qty). These are not independent.

There are a few possibilities regarding these products: perhaps the catalog information was never entered into the system, or perhaps only the catalog-related columns were lost during the ETL process.

Filling it with the modulo operator isn't correct because it leads to errors in an analysis like "most frequently sold category". Therefore, I should mark it as "unknown/missing". This way, the data isn't lost, and it's indicated that the product category is unknown. But I can't do that for the other columns as they are numerical. Therefore, I'm leaving them as they are.

In [18]:
products_clean.loc[
    products_clean["product_category_name"].isna()
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [19]:
products_clean['product_category_name'] = products_clean['product_category_name'].fillna('unknown')
products_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32951 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32945 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


So, I'm checking to see if these 610 products were actually sold. If they hadn't been sold and it wouldn't affect the analysis, then they could have been deleted.

In [20]:
missing_products = products_clean[
    products_clean["product_category_name"].isna()
]

missing_products.merge(
    order_items_clean,
    on="product_id",
    how="inner"
).shape

(0, 15)

The other empty values ​​in the products table are the columns I intentionally set to NaN beforehand (product_weight_g, product_length_cm, product_height_cm, product_width_cm).

I'm still reviewing these records. If the categories were the same, I might be able to fill them in accordingly, but the categories are different. Besides, the ratio is very small, so I'm leaving it as NaN.

In [21]:
products_clean.loc[
    products_clean["product_height_cm"].isna(),
    [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,unknown,NaN,NaN,NaN,NaN


I've also completed the products table. Next, I'm checking the empty values ​​in the order_reviews table. The review_comment_title and review_comment_message columns were empty. I'm checking if the review_score column is filled in these records. As you can see, people have given stars but haven't written reviews, which is quite normal. This happens on many sites like Amazon, Hepsiburada, and Trendyol. Since this is normal user behavior, I'm not touching it. Here, it's a case of Missing User Input, not Missing Data.

In [22]:
order_reviews_clean.loc[
    order_reviews_clean["review_comment_title"].isna(),
    "review_score"
].value_counts().sort_index()

review_score
1     9551
2     2673
3     7355
4    17407
5    50670
Name: count, dtype: int64

In [23]:
order_reviews_clean.loc[
    order_reviews_clean["review_comment_message"].isna(),
    "review_score"
].value_counts().sort_index()

review_score
1     2679
2     1006
3     4622
4    13166
5    36774
Name: count, dtype: int64

However, in these records, the payment_value is already 0, meaning no payment was ever made. Therefore, "payment_type = not_defined" is actually meaningful information. That's why I'm leaving it as is.

In [24]:
order_payments[order_payments['payment_type'] == 'not_defined']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


Some orders have multiple reviews. Before investigating this, I'm checking if the review_id is a reliable and unique identifier. Manual inspection of duplicate order_ids shows mixed patterns: some pairs share the same score and date (likely resubmissions), while others show sentiment shifting over time (e.g., a 3-star review followed by a 1-star complaint about an unresolved issue days later). This suggests taking the most recent review per order — rather than averaging — better reflects the customer's final, settled opinion.

In [25]:
dup_order_ids = order_reviews_clean['order_id'].value_counts()
dup_order_ids = dup_order_ids[dup_order_ids > 1].index

order_reviews_clean[order_reviews_clean['order_id'].isin(dup_order_ids)].sort_values(['order_id', 'review_creation_date']).head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25 00:00:00,2017-08-29 21:45:57
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21 00:00:00,2018-03-22 01:32:08
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29 00:00:00,2018-03-30 03:16:19
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11


Before removing duplicate records, I check if review_id is a trusted and unique key, and if any review_id is linked to more than one order_id.

In [26]:
review_id_order_counts = order_reviews.groupby('review_id')['order_id'].nunique()
review_id_order_counts[review_id_order_counts > 1]

review_id
00130cbe1f9d422698c812ed8ded1919    2
0115633a9c298b6a98bcbe4eee75345f    2
0174caf0ee5964646040cd94e15ac95e    2
017808d29fd1f942d97e50184dfb4c13    2
0254bd905dc677a6078990aad3331a36    2
                                   ..
fde2e6abaf5bb64f7407a44741c24dec    2
fde5986d35c89aa1b6ce4149de82a0d3    2
fe5c833752953fed3209646f1f63b53c    2
ff2fc9e68f8aabfbe18d710b83aabd30    2
ffb8cff872a625632ac983eb1f88843c    2
Name: order_id, Length: 789, dtype: int64

Finding: 789 review_ids are linked to two different order_ids. Manual inspection of a sample case shows these belong to genuinely different orders — different order_id, different customer_id — but with near-identical purchase timestamps (2 seconds apart) and identical review content, score, and dates.

This is too specific to be coincidental and appears to be a data quality artifact in the source dataset — review_id is NOT a reliable unique identifier here. As a result, deduplication is based solely on order_id, not review_id, to avoid incorrectly dropping valid review data belonging to a different real order.

In [27]:
sample_review_id = '00130cbe1f9d422698c812ed8ded1919'
order_reviews[order_reviews['review_id'] == sample_review_id]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23


In [28]:
order_ids_to_check = orders[orders['order_id'].isin(
    order_reviews[order_reviews['review_id'] == sample_review_id]['order_id']
)]
order_ids_to_check[['order_id', 'customer_id', 'order_purchase_timestamp']]

,order_id,customer_id,order_purchase_timestamp
28241,04a28263e085d399c97ae49e0b477efa,fef2e5e63da9f3e1dd89e8e319468657,2018-02-02 18:01:10
74048,dfcdfc43867d1c1381bfaf62d6b9c195,a7026133ddbd2e86c83ecd4dfa4dbe01,2018-02-02 18:01:08


Since review_id cannot be trusted as a unique key, deduplicating directly on order_id, keeping the most recent review by creation date — this best reflects the customer's final sentiment. Confirmed: rows (98673) match unique order_id count (98673) — each order now has exactly one review, safe to merge with the orders table without risk of row duplication.

In [29]:
order_reviews_clean = order_reviews_clean.sort_values('review_creation_date').drop_duplicates(subset='order_id', keep='last')

print("Rows:", order_reviews_clean.shape[0])
print("Unique order_id:", order_reviews_clean['order_id'].nunique())

Rows: 98673
Unique order_id: 98673


### Missing Value Decisions: marketing_leads & closed_deals

`marketing_leads['origin']`: 60 missing values ​​(0.75%). I left it as NaN because it's a negligible percentage, and there's no reliable way to extract the true source.

`closed_deals`: (`has_company`, `has_gtin`, `declared_product_catalog_size`) are missing in approximately 90% of the rows. I left it as NaN instead of filling it because filling would mean fabricating data for the vast majority of records.

I saved this clean data to the processed folder. This way I'm not working with raw_data and the pipeline is secure.

In [30]:
orders_clean.to_csv("../data/processed/orders_clean.csv", index=False)
products_clean.to_csv("../data/processed/products_clean.csv", index=False)
order_reviews_clean.to_csv("../data/processed/order_reviews_clean.csv", index=False)
customers_clean.to_csv("../data/processed/customers_clean.csv", index=False)
geolocation_clean.to_csv("../data/processed/geolocation_clean.csv", index=False)
order_items_clean.to_csv("../data/processed/order_items_clean.csv", index=False)
order_payments_clean.to_csv("../data/processed/order_payments_clean.csv", index=False)
sellers_clean.to_csv("../data/processed/sellers_clean.csv", index=False)
category_translation_clean.to_csv("../data/processed/category_translation_clean.csv", index=False)
closed_deals_clean.to_csv("../data/processed/closed_deals_clean.csv", index=False)
marketing_leads_clean.to_csv("../data/processed/marketing_leads_clean.csv", index=False)